# Raw sEMG+ACC Transformer — NinaPro DB7 Exercise B / E1 (labels 1–17)

This notebook uses the **Transformer architecture parameters reported in the MobileViT paper**, but **does not use T-EKIM or CI-2DR**.

The evaluation pipeline matches the latest subject-dependent Bi-LSTM notebook:

- **Exercise B / E1 only**
- local labels **1–17**
- all 22 DB7 subjects
- one independent model per subject (**subject-dependent / within-subject**)
- for each gesture: **4 complete repetitions train + 1 validation + 1 test**
- split complete repetitions before filtering, ACC alignment/resampling, windowing, normalization, or model fitting
- repetition-local EMG filtering
- repetition-local ACC interval extraction/resampling
- 100-ms boundary trim
- **400-ms window, 100-ms stride**
- separate **healthy/intact S01–S20** and **amputee S21–S22** reporting
- overall, healthy-only, and amputee-only **classwise accuracy** for all 17 gestures

## Transformer adaptation

The paper reports for its Transformer comparison model:

- PatchEmbed × 1
- Transformer Encoder × 4
- Dense × 1
- embedding dimension = **512**
- heads = **8**
- FFN dimension = **2048**

The paper's classification comparison originally used T-EKIM 2-D input. Because this version intentionally removes T-EKIM/CI-2DR, the same Transformer depth/width is adapted to the **raw multimodal 48-channel time series**:

`12 EMG + 36 ACC -> temporal patch embedding -> Transformer -> 17 classes`

So this is a **raw multimodal adaptation of the paper's Transformer configuration**, not a bit-for-bit reproduction of the paper's classification experiment.

In [1]:
import os, gc, time, json, warnings, random, re
from pathlib import Path
from math import gcd

import numpy as np
import pandas as pd

from scipy import io
from scipy.signal import (
    resample_poly, butter, sosfiltfilt, iirnotch, filtfilt
)

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.optim import AdamW

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

warnings.filterwarnings('ignore')
print('Imports done.')

Imports done.


## Configuration

In [2]:
def _find_kaggle_input() -> Path:
    base = Path('/kaggle/input')
    if not base.exists():
        return Path('/kaggle/input/ninapro-db7/Dataset')

    def _has_subjects(p):
        return p.is_dir() and any(
            c.is_dir() and c.name.lower().startswith('subject_')
            for c in p.iterdir()
        )

    def _search(root, depth=0):
        if depth > 5:
            return None
        if _has_subjects(root):
            return root
        try:
            for child in sorted(root.iterdir()):
                if child.is_dir():
                    found = _search(child, depth + 1)
                    if found is not None:
                        return found
        except PermissionError:
            pass
        return None

    found = _search(base)
    return found if found is not None else Path('/kaggle/input/ninapro-db7/Dataset')


class Config:
    KAGGLE_INPUT = _find_kaggle_input()
    KAGGLE_WORKING = (
        Path('/kaggle/working')
        if Path('/kaggle').exists()
        else Path.cwd() / 'transformer_raw_multimodal_working'
    )
    RUN_TAG = 'transformer_paper_raw_emg_acc_E1_B_1_17_subject_specific'
    DATA_CACHE_DIR = KAGGLE_WORKING / f'raw_file_cache_{RUN_TAG}'
    CKPT_DIR = KAGGLE_WORKING / f'ckpts_{RUN_TAG}'
    RESULTS_DIR = KAGGLE_WORKING / f'results_{RUN_TAG}'
    PLOT_DIR = KAGGLE_WORKING / f'plots_{RUN_TAG}'

    INTACT_SUBJECTS = list(range(1, 21))
    AMPUTEE_SUBJECTS = [21, 22]
    SUBJECTS = list(range(1, 23))
    RUN_SUBJECTS = SUBJECTS.copy()

    REPS_PER_GESTURE = 6
    TRAIN_REPS = 4
    VAL_REPS = 1
    TEST_REPS = 1

    EXERCISE_NAME = 'exerciseB_E1_labels1_17'
    EXERCISE_IDS = (1,)
    GESTURE_MIN = 1
    GESTURE_MAX = 17
    N_CLASSES = 17

    EMG_FS = 2000
    ACC_FS = 148
    TARGET_FS = 2000
    EMG_KEY = 'emg'
    ACC_KEY = 'acc'
    LBL_KEY = 'restimulus'
    N_EMG_CH = 12
    N_ACC_CH = 36
    N_INPUT_CH = N_EMG_CH + N_ACC_CH
    USE_ACC = True

    BANDPASS_LOW_HZ = 20.0
    BANDPASS_HIGH_HZ = 450.0
    FILTER_ORDER = 4
    NOTCH_HZ = 50.0
    NOTCH_Q = 30.0

    WIN_MS = 400
    STEP_MS = 100
    TRIM_MS = 100
    WIN_SAMPLES = int(WIN_MS * TARGET_FS / 1000)
    STEP_SAMPLES = int(STEP_MS * TARGET_FS / 1000)
    TRIM_SAMPLES = int(TRIM_MS * TARGET_FS / 1000)

    # Raw-time patch adaptation: 100 samples = 50 ms -> 8 tokens/window.
    # The paper does not specify a raw-time patch length because its classification
    # Transformer was evaluated with T-EKIM image input.
    PATCH_SAMPLES = 100
    PATCH_STRIDE = 100

    # Paper Transformer configuration
    EMBED_DIM = 512
    NUM_HEADS = 8
    FFN_DIM = 2048
    NUM_ENCODER_LAYERS = 4
    DROPOUT = 0.15

    MIN_EPOCHS = 15
    MAX_EPOCHS = 100
    PATIENCE = 12
    MIN_REFIT_EPOCHS = 8
    BATCH_SIZE = 32
    LR = 2e-4
    WEIGHT_DECAY = 1e-4
    LABEL_SMOOTHING = 0.03
    GRAD_CLIP = 1.0
    NUM_WORKERS = 2

    CACHE_VERSION = 'transformer_raw_E1_only_labels1_17_v1'

    SEED = 42
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


for d in [
    Config.DATA_CACHE_DIR,
    Config.CKPT_DIR,
    Config.RESULTS_DIR,
    Config.PLOT_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

random.seed(Config.SEED)
np.random.seed(Config.SEED)
torch.manual_seed(Config.SEED)
if Config.DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(Config.SEED)

assert Config.EXERCISE_IDS == (1,)
assert Config.GESTURE_MIN == 1
assert Config.GESTURE_MAX == 17
assert Config.N_CLASSES == 17
assert Config.TRAIN_REPS + Config.VAL_REPS + Config.TEST_REPS == 6
assert Config.WIN_SAMPLES == 800
assert Config.STEP_SAMPLES == 200
assert Config.N_INPUT_CH == 48
assert (Config.WIN_SAMPLES - Config.PATCH_SAMPLES) % Config.PATCH_STRIDE == 0

print(f'Device              : {Config.DEVICE}')
print('Exercise            : Exercise B / E1 only')
print('Gesture labels      : 1–17')
print('Protocol            : 4 train + 1 val + 1 test complete repetitions')
print('Subject dependence  : one independent Transformer per subject')
print('Input               : preprocessed EMG+ACC = 48 × 800')
print('T-EKIM / CI-2DR     : DISABLED')
print(
    'Transformer         : '
    f'embed={Config.EMBED_DIM}, heads={Config.NUM_HEADS}, '
    f'layers={Config.NUM_ENCODER_LAYERS}, ffn={Config.FFN_DIM}'
)

Device              : cuda
Exercise            : Exercise B / E1 only
Gesture labels      : 1–17
Protocol            : 4 train + 1 val + 1 test complete repetitions
Subject dependence  : one independent Transformer per subject
Input               : preprocessed EMG+ACC = 48 × 800
T-EKIM / CI-2DR     : DISABLED
Transformer         : embed=512, heads=8, layers=4, ffn=2048


## Repetition-local raw loader, EMG filtering, and ACC alignment

These are the same leakage-safe preprocessing components used by the latest Bi-LSTM notebook. The complete repetition split is decided before acausal EMG filtering or ACC resampling.

In [3]:
class RawEMGACCPreprocessor:
    """
    Load RAW EMG, RAW ACC and EMG-aligned labels from one NinaPro file.

    No filtering, ACC resampling, masking, feature extraction or windowing
    is performed here.
    """

    @staticmethod
    def _find_key(data, candidates):
        for candidate in candidates:
            for key in data:
                if key.lower() == candidate.lower():
                    return key
        return None

    @staticmethod
    def _time_major(array, expected_channels=None, name='signal'):
        array = np.asarray(array)

        if array.ndim == 1:
            array = array[:, None]
        if array.ndim != 2:
            raise RuntimeError(
                f'{name}: expected 2-D array, got {array.shape}.'
            )

        if expected_channels is not None:
            if array.shape[1] == expected_channels:
                return array
            if array.shape[0] == expected_channels:
                return array.T
            raise RuntimeError(
                f'{name}: neither dimension matches expected '
                f'{expected_channels} channels: {array.shape}.'
            )

        # ACC time dimension should be much larger than channel count.
        if array.shape[0] < array.shape[1] and array.shape[0] <= 128:
            array = array.T

        return array

    def apply(self, mat_path: Path):
        data = io.loadmat(str(mat_path))

        emg_key = self._find_key(data, [Config.EMG_KEY])
        acc_key = self._find_key(data, [Config.ACC_KEY])
        lbl_key = self._find_key(
            data,
            [Config.LBL_KEY, 'stimulus', 'label', 'labels'],
        )

        if emg_key is None:
            raise KeyError(f'No EMG key in {mat_path.name}.')
        if acc_key is None:
            raise KeyError(
                f'No ACC key in {mat_path.name}; '
                'this notebook requires EMG + ACC.'
            )
        if lbl_key is None:
            raise KeyError(f'No label key in {mat_path.name}.')

        emg = self._time_major(
            data[emg_key],
            expected_channels=Config.N_EMG_CH,
            name=f'{mat_path.name} EMG',
        ).astype(np.float32)

        acc = self._time_major(
            data[acc_key],
            expected_channels=None,
            name=f'{mat_path.name} ACC',
        ).astype(np.float32)

        labels = np.asarray(
            data[lbl_key]
        ).reshape(-1).astype(np.int32)

        n = min(len(emg), len(labels))
        emg = emg[:n]
        labels = labels[:n]

        if len(acc) < 2:
            raise RuntimeError(
                f'{mat_path.name}: ACC has only {len(acc)} samples.'
            )

        return emg, acc, labels


class RepetitionEMGFilter:
    """Zero-phase filtering applied to one already-assigned EMG repetition."""

    def __init__(self):
        nyquist = Config.EMG_FS / 2.0
        self.sos = butter(
            Config.FILTER_ORDER,
            [
                Config.BANDPASS_LOW_HZ / nyquist,
                Config.BANDPASS_HIGH_HZ / nyquist,
            ],
            btype='bandpass',
            output='sos',
        )
        self.b_notch, self.a_notch = iirnotch(
            Config.NOTCH_HZ / nyquist,
            Config.NOTCH_Q,
        )

    def apply(self, repetition_emg: np.ndarray) -> np.ndarray:
        repetition_emg = np.asarray(
            repetition_emg,
            dtype=np.float32,
        )

        if repetition_emg.ndim != 2:
            raise ValueError(
                f'Expected (time,channels), got {repetition_emg.shape}.'
            )
        if len(repetition_emg) < 64:
            raise RuntimeError(
                f'EMG repetition unexpectedly short: '
                f'{len(repetition_emg)} samples.'
            )

        filtered = sosfiltfilt(
            self.sos,
            repetition_emg,
            axis=0,
        )
        filtered = filtfilt(
            self.b_notch,
            self.a_notch,
            filtered,
            axis=0,
        )
        return filtered.astype(np.float32)


class RepetitionACCResampler:
    """
    Select the ACC interval matching one EMG repetition from the SAME
    source file, then resample only that interval to the EMG repetition length.
    """

    @staticmethod
    def extract_matching_interval(
        file_acc: np.ndarray,
        file_emg_length: int,
        emg_start: int,
        emg_end: int,
    ) -> np.ndarray:
        if not (
            0 <= emg_start < emg_end <= file_emg_length
        ):
            raise ValueError(
                f'Invalid EMG interval [{emg_start}, {emg_end}) '
                f'for file length {file_emg_length}.'
            )

        ratio = len(file_acc) / float(file_emg_length)

        # ceil keeps the selected interval inside the repetition boundary.
        acc_start = int(np.ceil(emg_start * ratio))
        acc_end = int(np.ceil(emg_end * ratio))

        acc_start = max(
            0,
            min(acc_start, len(file_acc) - 1),
        )
        acc_end = max(
            acc_start + 1,
            min(acc_end, len(file_acc)),
        )

        acc_rep = file_acc[
            acc_start:acc_end
        ].copy()

        if len(acc_rep) < 2:
            raise RuntimeError(
                f'Mapped ACC repetition is too short: '
                f'{len(acc_rep)} samples.'
            )

        return acc_rep

    @staticmethod
    def resample_to_emg_length(
        acc_rep: np.ndarray,
        target_length: int,
    ) -> np.ndarray:
        source_length = len(acc_rep)

        divisor = gcd(
            source_length,
            target_length,
        )
        up = target_length // divisor
        down = source_length // divisor

        resampled = resample_poly(
            acc_rep,
            up,
            down,
            axis=0,
        ).astype(np.float32)

        # Correct any one-sample numerical mismatch with samples from the
        # same repetition only.
        if len(resampled) > target_length:
            resampled = resampled[:target_length]
        elif len(resampled) < target_length:
            pad_count = target_length - len(resampled)
            pad = np.repeat(
                resampled[-1:, :],
                pad_count,
                axis=0,
            )
            resampled = np.vstack(
                [resampled, pad]
            )

        if len(resampled) != target_length:
            raise RuntimeError(
                f'ACC length mismatch: '
                f'{len(resampled)} != {target_length}.'
            )

        return resampled.astype(np.float32)


print(
    'Raw EMG+ACC loader, repetition-local EMG filter '
    'and repetition-local ACC resampler defined.'
)

Raw EMG+ACC loader, repetition-local EMG filter and repetition-local ACC resampler defined.


In [4]:
class SubjectLoader:
    def __init__(self):
        self.preprocessor = RawEMGACCPreprocessor()
        self.rep_filter = RepetitionEMGFilter()
        self.acc_resampler = RepetitionACCResampler()
        self.cache_dir = Config.DATA_CACHE_DIR

    def _ckpt_path(self, sid):
        exercise_tag = ''.join(
            f'E{exercise_id}'
            for exercise_id in Config.EXERCISE_IDS
        )
        return self.cache_dir / (
            f'RAW_FILES_S{sid:02d}_{exercise_tag}_'
            f'labels{Config.GESTURE_MIN}_{Config.GESTURE_MAX}_'
            f'emg_acc_{Config.CACHE_VERSION}.npz'
        )

    def _find_subject_dir(self, sid):
        candidates = [
            Config.KAGGLE_INPUT / f'Subject_{sid}',
            Config.KAGGLE_INPUT / f'subject_{sid}',
            Config.KAGGLE_INPUT / f'S{sid}',
            Config.KAGGLE_INPUT / f's{sid}',
        ]

        for path in candidates:
            if path.is_dir():
                return path

        for path in sorted(
            Config.KAGGLE_INPUT.iterdir()
        ):
            if (
                path.is_dir()
                and path.name.lower().endswith(str(sid))
            ):
                return path

        raise FileNotFoundError(
            f'Cannot find subject {sid} under {Config.KAGGLE_INPUT}.'
        )

    @staticmethod
    def _is_exercise_file(path: Path, exercise_id: int) -> bool:
        name = path.stem.upper()
        return bool(
            re.search(
                rf'(^|_)E{exercise_id}(_|$)',
                name,
            )
        )

    def _selected_files(self, sid):
        subject_dir = self._find_subject_dir(sid)
        all_mat_files = (
            sorted(subject_dir.glob('*.mat'))
            or sorted(subject_dir.rglob('*.mat'))
        )

        selected = []
        for exercise_id in Config.EXERCISE_IDS:
            matches = [
                path
                for path in all_mat_files
                if self._is_exercise_file(
                    path,
                    exercise_id,
                )
            ]

            if not matches:
                raise FileNotFoundError(
                    f'S{sid:02d}: missing E{exercise_id}. '
                    f'Available files: {[p.name for p in all_mat_files[:15]]}'
                )

            selected.extend(matches)

        # Hard guard: this notebook is Exercise B / E1 only.
        if len(selected) != 1:
            raise RuntimeError(
                f'S{sid:02d}: expected exactly one E1 file, '
                f'found {[p.name for p in selected]}.'
            )
        if not self._is_exercise_file(selected[0], 1):
            raise RuntimeError(
                f'S{sid:02d}: selected file is not E1: {selected[0].name}.'
            )
        return selected

    def _build_raw_file_cache(self, sid):
        mat_files = self._selected_files(sid)

        payload = {
            'n_files': np.array(
                len(mat_files),
                dtype=np.int64,
            ),
            'file_names_json': np.array(
                json.dumps(
                    [path.name for path in mat_files]
                )
            ),
        }

        acc_channel_counts = set()

        for index, mat_file in enumerate(mat_files):
            emg, acc, labels = (
                self.preprocessor.apply(mat_file)
            )

            if emg.shape[1] != Config.N_EMG_CH:
                raise RuntimeError(
                    f'S{sid:02d}, {mat_file.name}: '
                    f'expected {Config.N_EMG_CH} EMG channels, '
                    f'got {emg.shape[1]}.'
                )

            acc_channel_counts.add(
                int(acc.shape[1])
            )

            payload[f'emg_{index}'] = (
                emg.astype(np.float32)
            )
            payload[f'acc_{index}'] = (
                acc.astype(np.float32)
            )
            payload[f'labels_{index}'] = (
                labels.astype(np.int32)
            )

        if len(acc_channel_counts) != 1:
            raise RuntimeError(
                f'S{sid:02d}: inconsistent ACC channel counts '
                f'across E1/E2: {sorted(acc_channel_counts)}.'
            )

        n_acc_ch = int(
            next(iter(acc_channel_counts))
        )
        if n_acc_ch <= 0:
            raise RuntimeError(
                f'S{sid:02d}: no ACC channels found.'
            )

        payload['n_acc_ch'] = np.array(
            n_acc_ch,
            dtype=np.int64,
        )

        np.savez_compressed(
            self._ckpt_path(sid),
            **payload,
        )

        return n_acc_ch

    def _raw_cache_metadata(self, sid):
        with np.load(
            self._ckpt_path(sid),
            allow_pickle=False,
        ) as data:
            return (
                int(data['n_files']),
                int(data['n_acc_ch']),
                json.loads(
                    str(
                        data[
                            'file_names_json'
                        ].item()
                    )
                ),
            )

    def _load_raw_parts(self, sid):
        parts = []

        with np.load(
            self._ckpt_path(sid),
            allow_pickle=False,
        ) as data:
            n_files = int(data['n_files'])
            n_acc_ch = int(data['n_acc_ch'])
            file_names = json.loads(
                str(
                    data[
                        'file_names_json'
                    ].item()
                )
            )

            for index in range(n_files):
                parts.append({
                    'file_index': int(index),
                    'file_name': file_names[index],
                    'emg': data[
                        f'emg_{index}'
                    ].astype(np.float32),
                    'acc': data[
                        f'acc_{index}'
                    ].astype(np.float32),
                    'labels': data[
                        f'labels_{index}'
                    ].astype(np.int32),
                })

        return parts, n_acc_ch

    def process_all(self, subjects=None):
        subjects = subjects or Config.SUBJECTS
        available = []
        acc_counts = set()

        for sid in tqdm(
            subjects,
            desc='Loading RAW EMG+ACC subjects',
        ):
            checkpoint = self._ckpt_path(sid)

            if checkpoint.exists():
                (
                    n_files,
                    current_acc_ch,
                    _,
                ) = self._raw_cache_metadata(sid)
                print(
                    f'  S{sid:02d}: RAW per-file cache | '
                    f'files={n_files}, ACC={current_acc_ch} ch'
                )
            else:
                current_acc_ch = (
                    self._build_raw_file_cache(sid)
                )
                print(
                    f'  S{sid:02d}: saved RAW per-file '
                    f'EMG+ACC cache | ACC={current_acc_ch} ch'
                )

            if current_acc_ch <= 0:
                raise RuntimeError(
                    f'S{sid:02d}: ACC is required.'
                )

            acc_counts.add(
                int(current_acc_ch)
            )
            available.append(sid)

        if set(available) != set(subjects):
            missing = sorted(
                set(subjects) - set(available)
            )
            raise RuntimeError(
                f'Missing subjects: {missing}'
            )

        if len(acc_counts) != 1:
            raise RuntimeError(
                'ACC channel count differs across subjects: '
                f'{sorted(acc_counts)}'
            )

        return (
            available,
            int(next(iter(acc_counts))),
        )

    @staticmethod
    def _constant_label_runs(labels: np.ndarray):
        if len(labels) == 0:
            return

        boundaries = (
            np.flatnonzero(
                np.diff(labels) != 0
            )
            + 1
        )
        starts = np.r_[0, boundaries]
        ends = np.r_[boundaries, len(labels)]

        for start, end in zip(starts, ends):
            yield (
                int(start),
                int(end),
                int(labels[start]),
            )

    @staticmethod
    def _repetition_id(
        sid,
        gesture,
        repetition_index,
    ):
        return (
            int(sid),
            int(gesture),
            int(repetition_index + 1),
        )

    @staticmethod
    def _split_repetition_indices(
        sid,
        gesture,
    ):
        rng = np.random.default_rng(
            Config.SEED
            + 1009 * int(sid)
            + 9176 * int(gesture)
        )
        perm = rng.permutation(
            Config.REPS_PER_GESTURE
        ).tolist()

        test_idx = sorted(
            perm[:Config.TEST_REPS]
        )
        val_idx = sorted(
            perm[
                Config.TEST_REPS:
                Config.TEST_REPS + Config.VAL_REPS
            ]
        )
        train_idx = sorted(
            perm[
                Config.TEST_REPS
                + Config.VAL_REPS:
            ]
        )

        if len(train_idx) != Config.TRAIN_REPS:
            raise RuntimeError(
                'Unexpected number of training repetitions.'
            )

        return (
            train_idx,
            val_idx,
            test_idx,
        )

    def _collect_repetitions(self, parts):
        runs_by_gesture = {
            gesture: []
            for gesture in range(
                Config.GESTURE_MIN,
                Config.GESTURE_MAX + 1,
            )
        }

        for part_index, part in enumerate(parts):
            for (
                run_start,
                run_end,
                gesture,
            ) in self._constant_label_runs(
                part['labels']
            ):
                if gesture in runs_by_gesture:
                    runs_by_gesture[
                        gesture
                    ].append({
                        'part_index': int(
                            part_index
                        ),
                        'start': int(run_start),
                        'end': int(run_end),
                    })

        return runs_by_gesture


print(
    'Strict RAW Exercise-B / E1-only EMG+ACC subject loader defined.'
)

Strict RAW Exercise-B / E1-only EMG+ACC subject loader defined.


## Strict subject-specific raw windows

Each 400-ms window is created only after its complete repetition has already been assigned to train, validation, or test. No window crosses a repetition or split boundary.

In [5]:
def build_subject_repetition_windows(
    subject_loader: SubjectLoader,
    sid: int,
    n_acc_ch: int,
):
    """
    Strict leakage-safe subject-specific split.

    Ordering:
      raw Exercise-B / E1 EMG + ACC
        -> identify complete repetitions
        -> assign each repetition to train/val/test
        -> assert repetition-ID disjointness
        -> filter EMG inside each repetition
        -> select matching ACC interval from same file
        -> resample ACC inside that repetition
        -> concatenate EMG + ACC
        -> trim boundaries
        -> create overlapping windows

    Returns:
        result, assignment, provenance
    """
    parts, cached_n_acc = (
        subject_loader._load_raw_parts(sid)
    )

    if cached_n_acc != n_acc_ch:
        raise RuntimeError(
            f'S{sid:02d}: ACC channels {cached_n_acc} '
            f'!= expected {n_acc_ch}.'
        )

    runs_by_gesture = (
        subject_loader._collect_repetitions(parts)
    )

    split_X = {
        'train': [],
        'val': [],
        'test': [],
    }
    split_y = {
        'train': [],
        'val': [],
        'test': [],
    }
    split_counts = {
        split: np.zeros(
            Config.N_CLASSES,
            dtype=np.int64,
        )
        for split in split_X
    }
    split_rep_ids = {
        'train': set(),
        'val': set(),
        'test': set(),
    }

    assignment = {}
    expected_channels = (
        Config.N_EMG_CH + n_acc_ch
    )

    for gesture, runs in runs_by_gesture.items():
        if len(runs) != Config.REPS_PER_GESTURE:
            diagnostic = [
                (
                    parts[
                        record['part_index']
                    ]['file_name'],
                    record['start'],
                    record['end'],
                )
                for record in runs
            ]
            raise RuntimeError(
                f'S{sid:02d}, gesture {gesture}: expected '
                f'{Config.REPS_PER_GESTURE} complete repetitions, '
                f'found {len(runs)}. Runs={diagnostic}'
            )

        (
            train_idx,
            val_idx,
            test_idx,
        ) = (
            subject_loader
            ._split_repetition_indices(
                sid,
                gesture,
            )
        )

        assignment[str(gesture)] = {
            'train_reps': [
                int(i + 1)
                for i in train_idx
            ],
            'val_reps': [
                int(i + 1)
                for i in val_idx
            ],
            'test_reps': [
                int(i + 1)
                for i in test_idx
            ],
        }

        split_indices = {
            'train': train_idx,
            'val': val_idx,
            'test': test_idx,
        }
        class_id = (
            gesture - Config.GESTURE_MIN
        )

        # Register repetition identities before any transform.
        for (
            split_name,
            rep_indices,
        ) in split_indices.items():
            for rep_index in rep_indices:
                split_rep_ids[
                    split_name
                ].add(
                    subject_loader
                    ._repetition_id(
                        sid,
                        gesture,
                        rep_index,
                    )
                )

        assert (
            split_rep_ids['train']
            .isdisjoint(
                split_rep_ids['val']
            )
        )
        assert (
            split_rep_ids['train']
            .isdisjoint(
                split_rep_ids['test']
            )
        )
        assert (
            split_rep_ids['val']
            .isdisjoint(
                split_rep_ids['test']
            )
        )

        for (
            split_name,
            rep_indices,
        ) in split_indices.items():
            for rep_index in rep_indices:
                record = runs[rep_index]
                part = parts[
                    record['part_index']
                ]

                run_start = (
                    record['start']
                )
                run_end = (
                    record['end']
                )

                # RAW EMG repetition after split assignment.
                emg_raw = part[
                    'emg'
                ][
                    run_start:run_end
                ].copy()

                # Matching RAW ACC interval from the SAME source file.
                acc_raw = (
                    subject_loader
                    .acc_resampler
                    .extract_matching_interval(
                        file_acc=part['acc'],
                        file_emg_length=len(
                            part['emg']
                        ),
                        emg_start=run_start,
                        emg_end=run_end,
                    )
                )

                # Repetition-local transforms.
                emg_filtered = (
                    subject_loader
                    .rep_filter
                    .apply(emg_raw)
                )
                acc_resampled = (
                    subject_loader
                    .acc_resampler
                    .resample_to_emg_length(
                        acc_raw,
                        target_length=len(
                            emg_filtered
                        ),
                    )
                )

                if (
                    acc_resampled.shape[1]
                    != n_acc_ch
                ):
                    raise RuntimeError(
                        f'S{sid:02d}, gesture {gesture}, '
                        f'rep {rep_index + 1}: '
                        f'ACC channels={acc_resampled.shape[1]}, '
                        f'expected={n_acc_ch}.'
                    )

                signal = np.concatenate(
                    [
                        emg_filtered,
                        acc_resampled,
                    ],
                    axis=1,
                ).astype(np.float32)

                if (
                    signal.shape[1]
                    != expected_channels
                ):
                    raise RuntimeError(
                        f'S{sid:02d}: combined channel mismatch '
                        f'{signal.shape[1]} != '
                        f'{expected_channels}.'
                    )

                clean_start = (
                    Config.TRIM_SAMPLES
                )
                clean_end = (
                    len(signal)
                    - Config.TRIM_SAMPLES
                )

                if (
                    clean_end - clean_start
                    < Config.WIN_SAMPLES
                ):
                    raise RuntimeError(
                        f'S{sid:02d}, gesture {gesture}, '
                        f'rep {rep_index + 1}: '
                        'too short after boundary trim.'
                    )

                clean_signal = signal[
                    clean_start:clean_end
                ]

                # Overlapping windows never leave this one repetition/split.
                for window_start in range(
                    0,
                    len(clean_signal)
                    - Config.WIN_SAMPLES
                    + 1,
                    Config.STEP_SAMPLES,
                ):
                    segment = clean_signal[
                        window_start:
                        window_start
                        + Config.WIN_SAMPLES
                    ]

                    if (
                        segment.shape[0]
                        != Config.WIN_SAMPLES
                    ):
                        continue

                    split_X[
                        split_name
                    ].append(
                        segment.T.copy()
                    )
                    split_y[
                        split_name
                    ].append(
                        class_id
                    )
                    split_counts[
                        split_name
                    ][class_id] += 1

    # Final repetition-level disjointness proof.
    assert (
        split_rep_ids['train']
        .isdisjoint(
            split_rep_ids['val']
        )
    )
    assert (
        split_rep_ids['train']
        .isdisjoint(
            split_rep_ids['test']
        )
    )
    assert (
        split_rep_ids['val']
        .isdisjoint(
            split_rep_ids['test']
        )
    )

    expected_rep_ids = (
        Config.N_CLASSES
        * Config.REPS_PER_GESTURE
    )
    all_rep_ids = (
        split_rep_ids['train']
        | split_rep_ids['val']
        | split_rep_ids['test']
    )

    if len(all_rep_ids) != expected_rep_ids:
        raise RuntimeError(
            f'S{sid:02d}: expected '
            f'{expected_rep_ids} unique '
            f'repetition IDs, found '
            f'{len(all_rep_ids)}.'
        )

    result = {}

    for split_name in (
        'train',
        'val',
        'test',
    ):
        if not split_X[split_name]:
            raise RuntimeError(
                f'S{sid:02d}: no '
                f'{split_name} windows '
                'were created.'
            )

        X = np.stack(
            split_X[split_name]
        ).astype(np.float32)
        y = np.asarray(
            split_y[split_name],
            dtype=np.int64,
        )

        if X.shape[1] != expected_channels:
            raise RuntimeError(
                f'S{sid:02d}: '
                f'{split_name} expected '
                f'{expected_channels} channels, '
                f'got {X.shape[1]}.'
            )

        missing = np.flatnonzero(
            split_counts[
                split_name
            ] == 0
        )
        if len(missing):
            missing_gestures = [
                int(
                    i + Config.GESTURE_MIN
                )
                for i in missing
            ]
            raise RuntimeError(
                f'S{sid:02d}: '
                f'{split_name} missing '
                f'gestures {missing_gestures}.'
            )

        result[
            split_name
        ] = (
            X,
            y,
        )

    provenance = {
        split: [
            list(rep_id)
            for rep_id in sorted(
                split_rep_ids[split]
            )
        ]
        for split in (
            'train',
            'val',
            'test',
        )
    }

    del parts
    gc.collect()

    return (
        result,
        assignment,
        provenance,
    )

## Train-only raw-channel normalization

Mean and standard deviation are fitted separately for each raw signal channel from the current subject's training repetitions only. During final refit, they are freshly fitted on train+validation only.

In [6]:
def fit_channel_normalizer(*arrays):
    '''Fit one mean/std per channel over windows and time, without concatenating arrays.'''
    if not arrays:
        raise ValueError('At least one array is required.')

    sum_x = None
    sum_x2 = None
    count = 0

    for X in arrays:
        X = np.asarray(X, dtype=np.float64)
        if X.ndim != 3:
            raise ValueError(f'Expected (N,C,T), got {X.shape}.')

        local_sum = X.sum(axis=(0, 2))
        local_sum2 = np.square(X).sum(axis=(0, 2))
        n = X.shape[0] * X.shape[2]

        if sum_x is None:
            sum_x = local_sum
            sum_x2 = local_sum2
        else:
            sum_x += local_sum
            sum_x2 += local_sum2
        count += n

    mean = sum_x / count
    var = np.maximum(sum_x2 / count - np.square(mean), 1e-8)
    std = np.sqrt(var)
    return mean.astype(np.float32), std.astype(np.float32)


class RawWindowDataset(Dataset):
    def __init__(self, X, y, channel_mean, channel_std):
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.int64)
        self.mean = torch.from_numpy(channel_mean.astype(np.float32)).view(-1, 1)
        self.std = torch.from_numpy(channel_std.astype(np.float32)).view(-1, 1)

        if self.X.ndim != 3:
            raise ValueError(f'Expected X=(N,C,T), got {self.X.shape}.')
        if self.X.shape[1] != Config.N_INPUT_CH:
            raise ValueError(f'Expected {Config.N_INPUT_CH} channels, got {self.X.shape[1]}.')
        if self.X.shape[2] != Config.WIN_SAMPLES:
            raise ValueError(f'Expected {Config.WIN_SAMPLES} samples, got {self.X.shape[2]}.')

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx])
        x = (x - self.mean) / self.std
        return x, torch.tensor(self.y[idx], dtype=torch.long)


def make_loader(dataset, shuffle):
    return DataLoader(
        dataset,
        batch_size=Config.BATCH_SIZE,
        shuffle=shuffle,
        num_workers=Config.NUM_WORKERS,
        pin_memory=(Config.DEVICE.type == 'cuda'),
        drop_last=False,
    )

## Transformer from the paper — raw multimodal adaptation

Architecture parameters retained from the paper's Transformer comparison model:

- 1 PatchEmbed
- 4 Transformer encoders
- 512-dimensional embedding
- 8 attention heads
- 2048-dimensional FFN
- 1 dense classifier

**ACC is used.** The 12 EMG + 36 ACC channels are jointly embedded into every temporal patch token. There is no separate ACC MLP and no T-EKIM/CI-2DR.

In [7]:
class RawMultimodalTransformer(nn.Module):
    def __init__(self, n_classes=Config.N_CLASSES):
        super().__init__()

        self.patch_embed = nn.Conv1d(
            in_channels=Config.N_INPUT_CH,
            out_channels=Config.EMBED_DIM,
            kernel_size=Config.PATCH_SAMPLES,
            stride=Config.PATCH_STRIDE,
            bias=True,
        )

        self.n_patches = int(
            (Config.WIN_SAMPLES - Config.PATCH_SAMPLES)
            // Config.PATCH_STRIDE
            + 1
        )

        self.cls_token = nn.Parameter(torch.zeros(1, 1, Config.EMBED_DIM))
        self.pos_embed = nn.Parameter(
            torch.zeros(1, self.n_patches + 1, Config.EMBED_DIM)
        )
        self.input_dropout = nn.Dropout(Config.DROPOUT)

        layer = nn.TransformerEncoderLayer(
            d_model=Config.EMBED_DIM,
            nhead=Config.NUM_HEADS,
            dim_feedforward=Config.FFN_DIM,
            dropout=Config.DROPOUT,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=Config.NUM_ENCODER_LAYERS,
            norm=nn.LayerNorm(Config.EMBED_DIM),
        )
        self.head = nn.Linear(Config.EMBED_DIM, n_classes)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward_features(self, x):
        tokens = self.patch_embed(x).transpose(1, 2)
        cls = self.cls_token.expand(tokens.size(0), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        tokens = self.input_dropout(
            tokens + self.pos_embed[:, :tokens.size(1)]
        )
        encoded = self.encoder(tokens)
        return encoded[:, 0]

    def forward(self, x):
        return self.head(self.forward_features(x))

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


audit_model = RawMultimodalTransformer()
print(f'Patches/window   : {audit_model.n_patches}')
print(f'Trainable params : {audit_model.count_params():,}')
del audit_model

Patches/window   : 8
Trainable params : 15,082,513


## Training and evaluation

In [8]:
def evaluate_model(model, loader):
    model.eval()
    ys, logits_all = [], []
    total_loss = 0.0
    total_n = 0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for X, y in loader:
            X = X.to(Config.DEVICE, non_blocking=True)
            y = y.to(Config.DEVICE, non_blocking=True)
            logits = model(X)
            loss = criterion(logits, y)
            total_loss += float(loss.item()) * len(y)
            total_n += len(y)
            ys.append(y.cpu().numpy())
            logits_all.append(logits.cpu().numpy())

    y_true = np.concatenate(ys)
    logits = np.vstack(logits_all)
    proba = torch.softmax(torch.from_numpy(logits), dim=1).numpy()
    pred = proba.argmax(axis=1)

    return {
        'loss': total_loss / max(total_n, 1),
        'accuracy': float(accuracy_score(y_true, pred)),
        'balanced_accuracy': float(
            recall_score(y_true, pred, average='macro', zero_division=0)
        ),
        'precision_w': float(
            precision_score(y_true, pred, average='weighted', zero_division=0)
        ),
        'recall_w': float(
            recall_score(y_true, pred, average='weighted', zero_division=0)
        ),
        'f1_w': float(
            f1_score(y_true, pred, average='weighted', zero_division=0)
        ),
        'y_true': y_true,
        'proba': proba,
        'pred': pred,
    }


def safe_auc(y_true, proba):
    try:
        return float(
            roc_auc_score(
                y_true,
                proba,
                multi_class='ovr',
                average='weighted',
                labels=list(range(Config.N_CLASSES)),
            )
        )
    except Exception:
        return float('nan')


def train_for_selection(model, train_loader, val_loader, sid):
    model = model.to(Config.DEVICE)
    optimizer = AdamW(
        model.parameters(),
        lr=Config.LR,
        weight_decay=Config.WEIGHT_DECAY,
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=Config.LABEL_SMOOTHING)

    best_epoch = 1
    best_val_acc = -1.0
    best_val_loss = float('inf')
    best_state = None
    patience_count = 0
    history = []

    for epoch in range(1, Config.MAX_EPOCHS + 1):
        model.train()
        loss_sum = 0.0
        train_n = 0
        train_correct = 0

        for X, y in train_loader:
            X = X.to(Config.DEVICE, non_blocking=True)
            y = y.to(Config.DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(X)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP)
            optimizer.step()

            loss_sum += float(loss.item()) * len(y)
            train_n += len(y)
            train_correct += int((logits.argmax(dim=1) == y).sum().item())

        val = evaluate_model(model, val_loader)
        train_loss = loss_sum / max(train_n, 1)
        train_acc = train_correct / max(train_n, 1)

        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val['loss'],
            'val_acc': val['accuracy'],
        })

        improved = (
            val['accuracy'] > best_val_acc + 1e-8
            or (
                abs(val['accuracy'] - best_val_acc) <= 1e-8
                and val['loss'] < best_val_loss
            )
        )

        if improved:
            best_val_acc = val['accuracy']
            best_val_loss = val['loss']
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            patience_count = 0
        else:
            patience_count += 1

        if epoch >= Config.MIN_EPOCHS and patience_count >= Config.PATIENCE:
            break

    if best_state is None:
        raise RuntimeError(f'S{sid:02d}: no valid selection checkpoint.')

    model.load_state_dict(best_state)
    return model, int(best_epoch), pd.DataFrame(history), float(best_val_acc)


def train_fixed_epochs(model, loader, epochs):
    model = model.to(Config.DEVICE)
    optimizer = AdamW(
        model.parameters(),
        lr=Config.LR,
        weight_decay=Config.WEIGHT_DECAY,
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=Config.LABEL_SMOOTHING)

    for _ in range(int(epochs)):
        model.train()
        for X, y in loader:
            X = X.to(Config.DEVICE, non_blocking=True)
            y = y.to(Config.DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(X)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP)
            optimizer.step()

    return model

## One independent Transformer per subject

In [9]:
def run_one_subject(subject_loader, sid, n_acc_ch):
    print('\n' + '=' * 72)
    print(f'  SUBJECT S{sid:02d} — RAW EMG+ACC TRANSFORMER')
    print('=' * 72)

    raw_splits, assignment, provenance = build_subject_repetition_windows(
        subject_loader,
        sid,
        n_acc_ch,
    )

    X_train, y_train = raw_splits['train']
    X_val, y_val = raw_splits['val']
    X_test, y_test = raw_splits['test']

    if X_train.shape[1] != Config.N_INPUT_CH:
        raise RuntimeError(
            f'S{sid:02d}: expected {Config.N_INPUT_CH} channels, got {X_train.shape[1]}.'
        )

    print(f'  TRAIN: X={X_train.shape}, class counts={np.bincount(y_train, minlength=17).tolist()}')
    print(f'  VAL  : X={X_val.shape}, class counts={np.bincount(y_val, minlength=17).tolist()}')
    print(f'  TEST : X={X_test.shape}, class counts={np.bincount(y_test, minlength=17).tolist()}')

    # Selection normalizer = TRAIN only
    train_mean, train_std = fit_channel_normalizer(X_train)
    train_ds = RawWindowDataset(X_train, y_train, train_mean, train_std)
    val_ds = RawWindowDataset(X_val, y_val, train_mean, train_std)
    train_loader = make_loader(train_ds, shuffle=True)
    val_loader = make_loader(val_ds, shuffle=False)

    seed = Config.SEED + sid * 101
    torch.manual_seed(seed)
    if Config.DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(seed)

    selection_model = RawMultimodalTransformer()
    selection_model, best_epoch, history, best_val_acc = train_for_selection(
        selection_model,
        train_loader,
        val_loader,
        sid,
    )
    history.to_csv(
        Config.RESULTS_DIR / f'transformer_selection_history_S{sid:02d}.csv',
        index=False,
    )

    # Final normalizer = TRAIN + VAL only
    final_mean, final_std = fit_channel_normalizer(X_train, X_val)
    refit_train_ds = RawWindowDataset(X_train, y_train, final_mean, final_std)
    refit_val_ds = RawWindowDataset(X_val, y_val, final_mean, final_std)
    refit_ds = ConcatDataset([refit_train_ds, refit_val_ds])
    refit_loader = make_loader(refit_ds, shuffle=True)

    refit_epochs = max(
        Config.MIN_REFIT_EPOCHS,
        int(round(best_epoch * len(X_train) / (len(X_train) + len(X_val)))),
    )

    final_seed = seed + 100000
    torch.manual_seed(final_seed)
    if Config.DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(final_seed)

    final_model = RawMultimodalTransformer()
    final_model = train_fixed_epochs(final_model, refit_loader, refit_epochs)

    test_ds = RawWindowDataset(X_test, y_test, final_mean, final_std)
    test_loader = make_loader(test_ds, shuffle=False)
    test = evaluate_model(final_model, test_loader)

    metrics = {
        'subject': int(sid),
        'accuracy': float(test['accuracy']),
        'balanced_accuracy': float(test['balanced_accuracy']),
        'precision_w': float(test['precision_w']),
        'recall_w': float(test['recall_w']),
        'f1_w': float(test['f1_w']),
        'roc_auc_w': safe_auc(test['y_true'], test['proba']),
        'best_epoch': int(best_epoch),
        'best_val_accuracy': float(best_val_acc),
        'refit_epochs': int(refit_epochs),
        'train_size': int(len(X_train)),
        'val_size': int(len(X_val)),
        'test_size': int(len(X_test)),
        'input_channels': int(Config.N_INPUT_CH),
        'patches_per_window': int(final_model.n_patches),
        'parameter_count': int(final_model.count_params()),
        'tekim': False,
        'ci2dr': False,
    }

    print(
        f'  S{sid:02d} | Acc={metrics["accuracy"]:.4f} | '
        f'BalAcc={metrics["balanced_accuracy"]:.4f} | '
        f'F1={metrics["f1_w"]:.4f} | '
        f'best/refit={best_epoch}/{refit_epochs}'
    )

    y_true_out = test['y_true'].copy()
    proba_out = test['proba'].copy()

    del raw_splits, X_train, X_val, X_test
    del train_ds, val_ds, train_loader, val_loader, selection_model
    del refit_train_ds, refit_val_ds, refit_ds, refit_loader, final_model
    del test_ds, test_loader
    gc.collect()
    if Config.DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

    return metrics, y_true_out, proba_out, assignment, provenance

## Healthy/amputee and classwise reporting

In [10]:
def group_summary(name, subject_ids, subject_df, subject_order, all_y_true, all_y_proba):
    ids = set(int(s) for s in subject_ids)
    group_df = subject_df[subject_df['subject'].isin(ids)].copy()
    positions = [i for i, sid in enumerate(subject_order) if sid in ids]

    y_true = np.concatenate([all_y_true[i] for i in positions])
    proba = np.vstack([all_y_proba[i] for i in positions])
    pred = proba.argmax(axis=1)

    return {
        'group': name,
        'subjects': sorted(ids),
        'n_subjects': int(len(group_df)),
        'mean_subject_accuracy': float(group_df['accuracy'].mean()),
        'std_subject_accuracy': float(
            group_df['accuracy'].std(ddof=1) if len(group_df) > 1 else 0.0
        ),
        'mean_subject_balanced_accuracy': float(group_df['balanced_accuracy'].mean()),
        'std_subject_balanced_accuracy': float(
            group_df['balanced_accuracy'].std(ddof=1) if len(group_df) > 1 else 0.0
        ),
        'mean_subject_f1_w': float(group_df['f1_w'].mean()),
        'std_subject_f1_w': float(
            group_df['f1_w'].std(ddof=1) if len(group_df) > 1 else 0.0
        ),
        'pooled_test_accuracy': float(accuracy_score(y_true, pred)),
    }


def classwise_table(subject_order, all_y_true, all_y_proba):
    healthy = set(Config.INTACT_SUBJECTS)
    amputee = set(Config.AMPUTEE_SUBJECTS)
    rows = []

    for class_index in range(Config.N_CLASSES):
        gesture = class_index + Config.GESTURE_MIN
        row = {
            'gesture_label': int(gesture),
            'class_name': f'G{gesture:02d}',
        }

        for group_name, allowed in [
            ('overall', set(subject_order)),
            ('healthy', healthy),
            ('amputee', amputee),
        ]:
            correct = 0
            support = 0
            for sid, y_true, proba in zip(subject_order, all_y_true, all_y_proba):
                if sid not in allowed:
                    continue
                pred = proba.argmax(axis=1)
                mask = (y_true == class_index)
                support += int(mask.sum())
                correct += int(np.sum(pred[mask] == y_true[mask]))

            row[f'{group_name}_correct'] = int(correct)
            row[f'{group_name}_support'] = int(support)
            row[f'{group_name}_accuracy'] = (
                float(correct / support) if support > 0 else float('nan')
            )

        rows.append(row)

    return pd.DataFrame(rows)


def subject_by_class_table(subject_order, all_y_true, all_y_proba):
    rows = []
    for sid, y_true, proba in zip(subject_order, all_y_true, all_y_proba):
        pred = proba.argmax(axis=1)
        row = {
            'subject': int(sid),
            'group': 'amputee' if sid in Config.AMPUTEE_SUBJECTS else 'healthy',
        }

        for class_index in range(Config.N_CLASSES):
            gesture = class_index + 1
            mask = (y_true == class_index)
            support = int(mask.sum())
            row[f'G{gesture:02d}_support'] = support
            row[f'G{gesture:02d}_accuracy'] = (
                float(np.mean(pred[mask] == y_true[mask]))
                if support > 0 else float('nan')
            )
        rows.append(row)

    return pd.DataFrame(rows)

## Run all subjects

In [11]:
def main():
    print('\n' + '=' * 76)
    print(' RAW EMG+ACC TRANSFORMER — EXERCISE B / E1 — LABELS 1–17')
    print('=' * 76)
    print('T-EKIM / CI-2DR: NOT USED')
    print('ACC: all 36 channels are jointly used with the 12 EMG channels.')

    subject_loader = SubjectLoader()
    available_subjects, n_acc_ch = subject_loader.process_all(Config.RUN_SUBJECTS)

    if n_acc_ch != Config.N_ACC_CH:
        raise RuntimeError(
            f'Expected {Config.N_ACC_CH} ACC channels, detected {n_acc_ch}.'
        )

    all_metrics = []
    all_y_true = []
    all_y_proba = []
    assignments = {}
    provenance = {}
    subject_order = []

    for sid in Config.RUN_SUBJECTS:
        metrics, y_true, proba, assignment, subject_provenance = run_one_subject(
            subject_loader,
            sid,
            n_acc_ch,
        )
        all_metrics.append(metrics)
        all_y_true.append(y_true)
        all_y_proba.append(proba)
        assignments[str(sid)] = assignment
        provenance[str(sid)] = subject_provenance
        subject_order.append(int(sid))

        with open(
            Config.RESULTS_DIR / f'transformer_metrics_S{sid:02d}.json',
            'w',
        ) as f:
            json.dump(metrics, f, indent=2)

    subject_df = pd.DataFrame(all_metrics).sort_values('subject').reset_index(drop=True)

    pooled_true = np.concatenate(all_y_true)
    pooled_proba = np.vstack(all_y_proba)
    pooled_pred = pooled_proba.argmax(axis=1)

    aggregate = {
        'n_subject_models': int(len(subject_df)),
        'mean_subject_accuracy': float(subject_df['accuracy'].mean()),
        'std_subject_accuracy': float(subject_df['accuracy'].std(ddof=1)),
        'mean_subject_balanced_accuracy': float(subject_df['balanced_accuracy'].mean()),
        'std_subject_balanced_accuracy': float(subject_df['balanced_accuracy'].std(ddof=1)),
        'mean_subject_f1_w': float(subject_df['f1_w'].mean()),
        'std_subject_f1_w': float(subject_df['f1_w'].std(ddof=1)),
        'pooled_test_accuracy': float(accuracy_score(pooled_true, pooled_pred)),
        'protocol': 'subject-dependent within-subject, E1/Exercise-B labels1-17, 4/1/1',
        'input': 'filtered EMG12 + aligned/resampled ACC36 raw windows',
        'tekim': False,
        'ci2dr': False,
    }

    healthy_summary = group_summary(
        'healthy/intact (S01-S20)',
        Config.INTACT_SUBJECTS,
        subject_df,
        subject_order,
        all_y_true,
        all_y_proba,
    )
    amputee_summary = group_summary(
        'amputee (S21-S22)',
        Config.AMPUTEE_SUBJECTS,
        subject_df,
        subject_order,
        all_y_true,
        all_y_proba,
    )
    group_df = pd.DataFrame([healthy_summary, amputee_summary])
    classwise_df = classwise_table(subject_order, all_y_true, all_y_proba)
    subject_by_class_df = subject_by_class_table(subject_order, all_y_true, all_y_proba)

    subject_df.to_csv(Config.RESULTS_DIR / 'transformer_subject_results.csv', index=False)
    group_df.to_csv(Config.RESULTS_DIR / 'transformer_healthy_vs_amputee.csv', index=False)
    classwise_df.to_csv(
        Config.RESULTS_DIR / 'transformer_classwise_overall_healthy_amputee.csv',
        index=False,
    )
    subject_by_class_df.to_csv(
        Config.RESULTS_DIR / 'transformer_subject_by_class_accuracy.csv',
        index=False,
    )

    with open(Config.RESULTS_DIR / 'transformer_aggregate.json', 'w') as f:
        json.dump(aggregate, f, indent=2)
    with open(Config.RESULTS_DIR / 'transformer_repetition_assignments.json', 'w') as f:
        json.dump(assignments, f, indent=2)
    with open(Config.RESULTS_DIR / 'transformer_repetition_provenance.json', 'w') as f:
        json.dump(provenance, f, indent=2)

    print('\n' + '=' * 76)
    print(' PER-SUBJECT RESULTS')
    print('=' * 76)
    print(
        subject_df[
            ['subject','accuracy','balanced_accuracy','f1_w','roc_auc_w','best_epoch','refit_epochs']
        ].to_string(index=False)
    )

    print('\nAcross all subject-specific Transformer models:')
    print(
        f'Mean Accuracy          : {aggregate["mean_subject_accuracy"]:.4f} ± '
        f'{aggregate["std_subject_accuracy"]:.4f}'
    )
    print(
        f'Mean Balanced Accuracy : {aggregate["mean_subject_balanced_accuracy"]:.4f} ± '
        f'{aggregate["std_subject_balanced_accuracy"]:.4f}'
    )
    print(
        f'Mean F1 (weighted)     : {aggregate["mean_subject_f1_w"]:.4f} ± '
        f'{aggregate["std_subject_f1_w"]:.4f}'
    )
    print(f'Pooled Test Accuracy   : {aggregate["pooled_test_accuracy"]:.4f}')

    print('\n' + '=' * 76)
    print(' HEALTHY / INTACT vs AMPUTEE')
    print('=' * 76)
    for _, row in group_df.iterrows():
        print(f'\n{row["group"]}:')
        print(
            f'  Mean Accuracy        : {row["mean_subject_accuracy"]:.4f} ± '
            f'{row["std_subject_accuracy"]:.4f}'
        )
        print(
            f'  Mean Balanced Acc    : {row["mean_subject_balanced_accuracy"]:.4f} ± '
            f'{row["std_subject_balanced_accuracy"]:.4f}'
        )
        print(
            f'  Mean F1              : {row["mean_subject_f1_w"]:.4f} ± '
            f'{row["std_subject_f1_w"]:.4f}'
        )
        print(f'  Pooled Test Accuracy : {row["pooled_test_accuracy"]:.4f}')

    print('\n' + '=' * 76)
    print(' CLASSWISE TEST ACCURACY')
    print('=' * 76)
    print(
        classwise_df[
            [
                'gesture_label',
                'overall_accuracy',
                'healthy_accuracy',
                'amputee_accuracy',
                'overall_support',
                'healthy_support',
                'amputee_support',
            ]
        ].to_string(index=False, float_format=lambda x: f'{x:.4f}')
    )

    print('\nSaved files:')
    print('  transformer_subject_results.csv')
    print('  transformer_healthy_vs_amputee.csv')
    print('  transformer_classwise_overall_healthy_amputee.csv')
    print('  transformer_subject_by_class_accuracy.csv')
    print('  transformer_aggregate.json')
    print('  transformer_repetition_assignments.json')
    print('  transformer_repetition_provenance.json')

    print('\nLeakage controls:')
    print('  ✓ E1 / Exercise B only')
    print('  ✓ labels 1–17 only')
    print('  ✓ complete repetitions split before filtering/resampling/windowing')
    print('  ✓ EMG zero-phase filtering is repetition-local')
    print('  ✓ ACC interval/resampling is repetition-local')
    print('  ✓ overlapping windows never cross repetition/split boundaries')
    print('  ✓ raw channel normalization fitted on TRAIN only during selection')
    print('  ✓ final normalizer fitted on TRAIN+VAL only')
    print('  ✓ test untouched until final evaluation')
    print('  ✓ T-EKIM / CI-2DR not used')

    return subject_df, aggregate, group_df, classwise_df, subject_by_class_df


(
    per_subject_results,
    aggregate_results,
    group_summary_results,
    classwise_accuracy_results,
    subject_by_class_results,
) = main()


 RAW EMG+ACC TRANSFORMER — EXERCISE B / E1 — LABELS 1–17
T-EKIM / CI-2DR: NOT USED
ACC: all 36 channels are jointly used with the 12 EMG channels.


Loading RAW EMG+ACC subjects:   0%|          | 0/22 [00:00<?, ?it/s]

  S01: saved RAW per-file EMG+ACC cache | ACC=36 ch
  S02: saved RAW per-file EMG+ACC cache | ACC=36 ch
  S03: saved RAW per-file EMG+ACC cache | ACC=36 ch
  S04: saved RAW per-file EMG+ACC cache | ACC=36 ch
  S05: saved RAW per-file EMG+ACC cache | ACC=36 ch
  S06: saved RAW per-file EMG+ACC cache | ACC=36 ch
  S07: saved RAW per-file EMG+ACC cache | ACC=36 ch
  S08: saved RAW per-file EMG+ACC cache | ACC=36 ch
  S09: saved RAW per-file EMG+ACC cache | ACC=36 ch
  S10: saved RAW per-file EMG+ACC cache | ACC=36 ch
  S11: saved RAW per-file EMG+ACC cache | ACC=36 ch
  S12: saved RAW per-file EMG+ACC cache | ACC=36 ch
  S13: saved RAW per-file EMG+ACC cache | ACC=36 ch
  S14: saved RAW per-file EMG+ACC cache | ACC=36 ch
  S15: saved RAW per-file EMG+ACC cache | ACC=36 ch
  S16: saved RAW per-file EMG+ACC cache | ACC=36 ch
  S17: saved RAW per-file EMG+ACC cache | ACC=36 ch
  S18: saved RAW per-file EMG+ACC cache | ACC=36 ch
  S19: saved RAW per-file EMG+ACC cache | ACC=36 ch
  S20: saved

## Reporting note

Use the name:

**Raw multimodal Transformer adapted from the paper's Transformer configuration**

Do **not** call it an exact reproduction of the paper's Transformer classification experiment, because the paper's classification comparison used T-EKIM 2-D input, while this requested version directly tokenizes the raw preprocessed EMG+ACC time series.

Classwise accuracy is `correct test windows in a class / all test windows in that class`, equivalent to per-class recall.